# Calculate and visualize reference activation levels for WM, Emotion, and Language tasks

For information on how to use this notebook along with the original `.ipynb` file, please check the corresponding [Python notebook in the preprocessing reports repository from Dr. Raúl Rodriguez Cruces](https://github.com/rcruces/preproc_reports/blob/main/notebooks/2024_neurosynth-ICBM152-fsLR.ipynb).

## Setup

In [ ]:
!pip install nilearn --quiet
!pip install nibabel --quiet
!pip install brainspace --quiet
!pip install seaborn --quiet

import os
import tempfile
import requests
import numpy as np
import nibabel as nib
import seaborn as sns
import matplotlib.pyplot as plt

from nilearn import plotting, datasets
from brainspace.utils.parcellation import map_to_labels

1. Download the MNI atlases
  > **ICBM 2009a Nonlinear Asymmetric template** - 1×1x1mm template which includes T1w,T2w,PDw modalities, and tissue probabilities maps. Intensity inhomogeneity was performed using N3 version 1.10.1. Also included brain mask, eye mask and face mask.

  > **ICBM 2009c Nonlinear Asymmetric template** - 1×1x1mm template which includes T1w,T2w,PDw modalities, and tissue probabilities maps. Intensity inhomogeneity was performed using N3 version 1.11 Also included brain mask, eye mask and face mask.Sampling is different from 2009a template.

2. Multiply the white matter probability mask by 20 and add it to the T1 to enhance the WM contrast to facilitate the segmentation of `fastsurfer`
  > Example:
  ```
  !fslmaths mni_icbm152_wm_tal_nlin_asym_09a.nii -mul 20 -add mni_icbm152_t1_tal_nlin_asym_09a.nii mni_icbm152_asym_A.nii.gz
  ```

3. Then, run `fastsurfer` singularity container on the atlas.



In [ ]:
!fslmaths mni_icbm152_wm_tal_nlin_asym_09a.nii -mul 20 -add mni_icbm152_t1_tal_nlin_asym_09a.nii mni_icbm152_asym_A.nii.gz

# Number of threads
threads=15
# Image path
fastsurfer_img=fastsurfer-cpu-v2.2.0.sif
# Temporary directory path
export TMPDIR=/tmp/tmpfiles
# Freesurfer license
fs_license=/freesurfer-7.3.2/license.txt
# Path to outputs
SUBJECTS_DIR=/out/fastsurfer
# Subject ID
idBIDS=mni_icbm152_asym_C
# MRI|IMG to process
t1=mni_icbm152_asym_C.nii.gz

# Run the singularity container
singularity exec --writable-tmpfs --containall \
             -B "${SUBJECTS_DIR}":/output \
             -B "${TMPDIR}":/tmpdir \
             -B "${t1}":/tmpdir/${idBIDS}_T1w.nii.gz \
             -B "${fs_license}":/output/license.txt \
             "${fastsurfer_img}" \
                 /fastsurfer/run_fastsurfer.sh \
                 --fs_license /output/license.txt \
                 --t1 /tmpdir/${idBIDS}_T1w.nii.gz \
                 --sid "${idBIDS}" --sd /output --no_fs_T1 \
                 --parallel --threads "${threads}"

### Download the `anatomical.nii.gz` files from Neurosynth
We downloaded `association-test_z_FDR_0.01.nii.gz` files corresponding to our target tasks from the following links:
* [Working Memory](https://neurosynth.org/analyses/terms/working%20memory/)
* Emotion subtasks:
  * [Neutral](https://neurosynth.org/analyses/terms/neutral/)
  * [Fear](https://neurosynth.org/analyses/terms/fear/)
* [Language](https://neurosynth.org/analyses/terms/language/)